# Default notebook

This default notebook is executed using a Lakeflow job as defined in resources/sample_job.job.yml.

In [0]:
%run ./utils/logger

In [0]:
run_id = get_run_id()
print(run_id)

In [0]:
# Set default catalog and schema
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

In [0]:
products_df = spark.sql(f"""
SELECT *, (_metadata.file_name) as file_name
FROM read_files(
    'abfss://ecommerce@dataprojectadls.dfs.core.windows.net/{catalog}/inbound/products*.csv',
    format => 'csv',
    inferSchema => true
)
""")

In [0]:
display(products_df)

In [0]:
files_recieved = products_df.count()

if files_recieved > 0:
    print(f'Number of files recieved :{files_recieved}')
else:
    print('No files present in adls path')

In [0]:
try:
    spark.sql(f""" create table if not exists {catalog}.{schema}.products_stage
              as
              SELECT *, (_metadata.file_name) as file_name
              FROM read_files(
                  'abfss://ecommerce@dataprojectadls.dfs.core.windows.net/{catalog}/inbound/products*.csv',format => 'csv',
                  inferSchema => true
                  )
                  """)

    log_run(run_id, "products_ingest_pipeline", "products_raw", "SUCCESS", "products raw data satge load completed")

except Exception as e:
    log_run(run_id, "products_ingest_pipeline", "products_raw", "FAILED", error_message=str(e))
    raise 

In [0]:
%sql
describe table extended products_stage

In [0]:
%sql
describe table extended products

In [0]:
try:

    spark.sql(f"""
        TRUNCATE TABLE {catalog}.bronze.products
    """)

    spark.sql(f"""
        INSERT INTO {catalog}.bronze.products
        SELECT
            CAST(product_id AS STRING)       AS product_id,
            TRIM(product_name)               AS product_name,
            TRIM(category)                   AS category,
            TRIM(brand)                      AS brand,
            CURRENT_TIMESTAMP()              AS ingestion_time,
            file_name                        AS source_file
        FROM {catalog}.bronze.products_stage
    """)

    log_run(
        run_id,
        "products_ingest_pipeline",
        "bronze_load",
        "SUCCESS",
        "Products bronze load completed"
    )

except Exception as e:

    log_run(
        run_id,
        "products_ingest_pipeline",
        "bronze_load",
        "FAILED",
        error_message=str(e)
    )

    raise

In [0]:
%sql
DROP TABLE commerce_raw_dev.bronze.products_stage;